# Hunyuan3D-2：正面圖批次生成 3D（Colab T4）

把圖放進 Google Drive 的 `DRIVE_ROOT/in`，依序執行區塊 1～4。檔名的第一個字決定生成方式：

| 檔名 | 生成方式 |
|---|---|
| `character-<id>-front.png`、`monster-<id>-front.png` | 由正面圖補出左、背、右視角，再用多視角模型建形狀（角色和怪物設定相同） |
| `armor-<id>-stuff.png`、`appearance-<id>-stuff.png`、`weapon-<id>-stuff.png` | 只用這一張圖，交給單張圖專用的形狀模型；面數比角色少，解析度一樣 |
| `object-<id>-building.png` | 小物件（路燈、石頭）：只用這一張圖，一般精度 |
| `bigobject-<id>-front.png` ＋ `-back`、`-left`、`-right` | 大型建築（城門、教堂）：**四張圖都要你提供**，缺一張就等待跳過；最高精度（形狀網格 512、20 萬面、4K 貼圖），T4 放不下時自動降一級重試 |

**多視角的圖**：bigobject 必須放齊 `<id>-front`、`-back`、`-left`、`-right` 四張；character、monster 可以另外放 `-left`、`-back`、`-right`（選用），例如 `character-novice_female-back.png`。有放的角度直接採用，沒放的才由 MV-Adapter 生成。**left 圖裡主體的正面朝向畫面左邊，right 圖則朝向右邊**（left 是面對正面時往你的右手邊繞 90 度看到的樣子）。

區塊 4 會一張接一張處理 `in/` 裡所有未使用的正面圖；開著時你繼續丟新圖，它也會接著做，閒置一段時間後自動停止。

**固定執行環境（必做）**：「執行階段 → 變更執行階段類型」選 **T4 GPU**，「執行階段版本」選 **2026.07**。

| 區塊 | 做什麼 |
|---|---|
| 1 | 檢查 runtime 版本、GPU、記憶體 |
| 2 | 設定（Drive 路徑、光頭模式、模型、批次等待時間） |
| 3 | 安裝固定版本套件與流程程式（新 runtime 約 3～5 分鐘，不需重啟） |
| 4 | 批次生成：視角 → 形狀 → 貼圖 → 輸出，每個階段在獨立程序執行，結束時記憶體全部釋放 |
| 5 | 檢視結果、打包下載（選用） |

**輸出**（`DRIVE_ROOT/out`）：`hunyuan3d-<類別>-<id>.glb`（角色縮放到身高 1.65 m，其他保留原尺寸；腳踩地面、PBR 材質）、`.json`（授權與生成參數）、`-views.jpg`（四視角預覽）。檔名格式符合 RagnarokChange 的投遞規則。

**光頭模式（預設開啟，只套用在 character）**：RagnarokChange 的頭和頭髮是另外的素材，所以角色身體做成光頭最好處理。用 Gemini 產生正面圖時請加上：*「same character and outfit, completely bald head with no hair, no helmet, no backpack, no weapons, full body front view, A-pose, plain white background」*。正面圖是光頭時，其他視角才會一致。

In [ ]:
# @title 1. Check runtime version, GPU, RAM and disk
import os
# Reduce CUDA memory fragmentation; must be set before torch initialises CUDA.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import platform, shutil, subprocess, sys, torch

# Pinned Colab runtime: Runtime > Change runtime type > Runtime version > 2026.07.
EXPECTED_PYTHON = (3, 12)
EXPECTED_TORCH = "2.11."
print(f"Python: {platform.python_version()}")
print(f"Torch : {torch.__version__}  CUDA {torch.version.cuda}")
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook is pinned to Colab runtime version 2026.07 (Python 3.12). "
        "Runtime > Change runtime type > Runtime version > 2026.07, then run block 1 again.")
if not torch.__version__.startswith(EXPECTED_TORCH):
    print(f"WARNING: runtime 2026.07 ships torch {EXPECTED_TORCH}x; this runtime has {torch.__version__}. Pinned packages may not match.")

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
props = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability(0)
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gb = shutil.disk_usage("/content").free / 1024**3
print(f"GPU   : {props.name}  VRAM {props.total_memory / 1024**3:.1f} GB  sm_{cap[0]}{cap[1]}")
print(f"RAM   : {ram_gb:.1f} GB")
print(f"Disk  : {disk_gb:.0f} GB free")
nvcc = shutil.which("nvcc")
print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1] if nvcc else "nvcc: not found")
if props.total_memory / 1024**3 < 14:
    print("WARNING: less than 14 GB VRAM; set VIEW_COUNT = 4 and RENDER_SIZE = 1024.")

In [ ]:
# @title 2. Settings
# @markdown Model weights (about 42 GB) and compiled extensions stay in Drive.
DRIVE_ROOT = "/content/drive/MyDrive/RagnarokChange/hunyuan3d"  # @param {type:"string"}
# bald: characters (not monsters) are made without hair; RagnarokChange adds head and hair separately.
HEAD_MODE = "bald"  # @param ["bald", "keep"]
VIEW_COUNT = 4  # @param [4, 6] {type:"raw"}
VIEW_SEED = 1234  # @param {type:"integer"}
# Full model: better detail, about 3 min per shape on T4. Turbo: about 30 s.
SHAPE_MODEL = "hunyuan3d-dit-v2-mv"  # @param ["hunyuan3d-dit-v2-mv", "hunyuan3d-dit-v2-mv-turbo"]
# Shape model for armor / appearance / weapon (one image, Hunyuan3D-2).
SINGLE_SHAPE_MODEL = "hunyuan3d-dit-v2-0"  # @param ["hunyuan3d-dit-v2-0", "hunyuan3d-dit-v2-0-turbo"]
SHAPE_SEED = 1234  # @param {type:"integer"}
PAINT_MODEL = "hunyuan3d-paint-v2-0"  # @param ["hunyuan3d-paint-v2-0", "hunyuan3d-paint-v2-0-turbo"]
TEXTURE_SIZE = 2048  # @param [1024, 2048] {type:"raw"}
# Batch: check Drive/in every POLL_MINUTES; stop after IDLE_STOP_MINUTES without new images.
WATCH = True  # @param {type:"boolean"}
POLL_MINUTES = 2  # @param {type:"integer"}
IDLE_STOP_MINUTES = 30  # @param {type:"integer"}
# Code version of the pipeline package (this repository's branch).
PIPELINE_REF = "claude/gallant-thompson-py38mf"  # @param {type:"string"}
ESTIMATED_CACHE_GB = 42

import os
import shutil
from google.colab import drive
drive.mount("/content/drive")

PIPELINE_DIR = "/content/colab-ImageTo3D"
CONFIG_PATH = "/content/imageto3d_config.json"
HF_HOME = os.path.join(DRIVE_ROOT, "hf_cache")
REMBG_HOME = os.path.join(DRIVE_ROOT, "rembg_cache")
WHEEL_CACHE = os.path.join(DRIVE_ROOT, "wheels")
for name in ("in", "out", "work", "hf_cache", "rembg_cache", "wheels"):
    os.makedirs(os.path.join(DRIVE_ROOT, name), exist_ok=True)

# Model weights already in the Drive cache (deduplicated by inode).
seen, cache_bytes = set(), 0
for root, _, names in os.walk(os.path.join(HF_HOME, "hub")):
    for name in names:
        try:
            stat = os.stat(os.path.join(root, name))
        except OSError:
            continue
        if (stat.st_dev, stat.st_ino) not in seen:
            seen.add((stat.st_dev, stat.st_ino))
            cache_bytes += stat.st_size
free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1024**3
needed_gb = max(0, ESTIMATED_CACHE_GB - cache_bytes / 1024**3)
print(f"Drive free: {free_gb:.1f} GB | cached models: {cache_bytes / 1024**3:.1f} GB | still to download: about {needed_gb:.1f} GB")
if free_gb < needed_gb:
    raise RuntimeError(f"Not enough Drive space: free {needed_gb - free_gb:.1f} GB more.")

try:
    from google.colab import userdata
    if userdata.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

SETTINGS = dict(
    drive_root=DRIVE_ROOT, hf_home=HF_HOME, rembg_home=REMBG_HOME,
    head_mode=HEAD_MODE, view_count=VIEW_COUNT, view_seed=VIEW_SEED,
    shape_model=SHAPE_MODEL, single_shape_model=SINGLE_SHAPE_MODEL, shape_seed=SHAPE_SEED, paint_model=PAINT_MODEL, texture_size=TEXTURE_SIZE,
    poll_seconds=POLL_MINUTES * 60, idle_stop_minutes=IDLE_STOP_MINUTES,
)
print("Settings ready.")

In [ ]:
# @title 3. Install (do not stop it while it runs)
# @markdown A new runtime installs pinned packages and reuses CUDA extensions cached in Drive. No restart needed:
# @markdown the heavy work runs in separate processes that start after the install.
if "SETTINGS" not in globals():
    raise RuntimeError("Settings are missing. Run blocks 1 and 2 first.")
import glob, os, shutil, subprocess, sys, sysconfig

REPO = "/content/Hunyuan3D-2"
MV_REPO = "/content/MV-Adapter"
HUNYUAN_COMMIT = "f8db63096c8282cb27354314d896feba5ba6ff8a"
MV_COMMIT = "4277e0018232bac82bb2c103caf0893cedb711be"
MARK = "/content/.imageto3d_installed_v3"
CONSTRAINTS = "/content/constraints.txt"
# Versions verified against Colab runtime 2026.07 (Python 3.12, torch 2.11, numpy 2.0.2).
PINNED = [
    "transformers==4.57.3", "diffusers==0.34.0", "huggingface_hub==0.36.0", "accelerate==1.10.1",
    "peft==0.17.1", "omegaconf==2.3.0", "einops==0.8.1", "pymeshlab==2025.7.post1", "xatlas==0.0.11",
    "pygltflib==1.16.5", "trimesh==4.8.3", "onnxruntime==1.22.1", "pybind11==3.0.1", "ninja==1.13.0",
    "pooch==1.8.2", "jsonschema==4.25.1",
    # MV-Adapter imports these at module load (mvadapter/utils/typing.py).
    "jaxtyping==0.3.2", "typeguard==4.4.4",
]
PINNED_NO_DEPS = ["rembg==2.0.67", "pymatting==1.1.14"]
NVDIFFRAST = "git+https://github.com/NVlabs/nvdiffrast.git@v0.3.3"


def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    process = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in process.stdout:
        tail = (tail + [line])[-40:]
        if not line.startswith(("  ", "Requirement already")):
            print(line, end="", flush=True)
    if process.wait() != 0:
        print("".join(tail))
        raise RuntimeError(f"Command failed: {cmd}")


# Pipeline code: always refresh so fixes pushed to the branch apply on the next run.
if not os.path.exists(PIPELINE_DIR):
    sh(f"git clone https://github.com/chousir/colab-ImageTo3D.git {PIPELINE_DIR}")
sh(f"git fetch -q origin {PIPELINE_REF} && git checkout -q -B run FETCH_HEAD", cwd=PIPELINE_DIR)

if not os.path.exists(MARK):
    import torch
    capability = torch.cuda.get_device_capability(0)
    abi_tag = f"py{sys.version_info[0]}{sys.version_info[1]}-torch{torch.__version__.replace('+', '_')}-sm{capability[0]}{capability[1]}"
    wheel_dir = os.path.join(WHEEL_CACHE, abi_tag)
    os.makedirs(wheel_dir, exist_ok=True)

    # Keep Colab's compiled stack; a different torch or numpy would break CUDA extensions.
    frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    keep = ("numpy==", "scipy==", "torch==", "torchvision==", "triton==", "numba==", "llvmlite==",
            "pillow==", "opencv-python==", "opencv-python-headless==", "scikit-image==")
    with open(CONSTRAINTS, "w") as file:
        file.write("".join(line + "\n" for line in frozen.splitlines() if line.lower().startswith(keep)))

    if not os.path.exists(REPO):
        sh(f"git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git {REPO}")
    sh(f"git checkout -q {HUNYUAN_COMMIT}", cwd=REPO)
    if not os.path.exists(MV_REPO):
        sh(f"git clone https://github.com/huanngzh/MV-Adapter.git {MV_REPO}")
    sh(f"git checkout -q {MV_COMMIT}", cwd=MV_REPO)

    sh(f"pip install -c {CONSTRAINTS} " + " ".join(f"'{package}'" for package in PINNED))
    sh(f"pip install -c {CONSTRAINTS} --no-deps " + " ".join(PINNED_NO_DEPS))
    sh(f"pip install --no-deps -e {REPO}")
    # nvdiffrast 0.3.x compiles lazily and is only imported by MV-Adapter, so this install is quick.
    sh(f"pip install --no-build-isolation --no-deps {NVDIFFRAST}")

    # custom_rasterizer: build once per Python/torch/GPU combination, then reuse the wheel from Drive.
    wheels = sorted(glob.glob(os.path.join(wheel_dir, "custom_rasterizer-*.whl")))
    if not wheels:
        os.environ["TORCH_CUDA_ARCH_LIST"] = f"{capability[0]}.{capability[1]}"
        build_dir = "/content/wheel_build"
        shutil.rmtree(build_dir, ignore_errors=True)
        sh(f"pip wheel -c {CONSTRAINTS} --no-build-isolation --no-deps -w {build_dir} .",
           cwd=f"{REPO}/hy3dgen/texgen/custom_rasterizer")
        for wheel in glob.glob(os.path.join(build_dir, "custom_rasterizer-*.whl")):
            shutil.copy2(wheel, wheel_dir)
        wheels = sorted(glob.glob(os.path.join(wheel_dir, "custom_rasterizer-*.whl")))
    else:
        print("Using cached wheel:", wheels[-1])
    sh(f"pip install --no-deps --force-reinstall '{wheels[-1]}'")

    renderer_dir = f"{REPO}/hy3dgen/texgen/differentiable_renderer"
    mesh_processor = f"mesh_processor{sysconfig.get_config_var('EXT_SUFFIX')}"
    cached_processor = os.path.join(wheel_dir, mesh_processor)
    if not os.path.exists(cached_processor):
        try:
            sh(f"c++ -O3 -Wall -shared -std=c++11 -fPIC $({sys.executable} -m pybind11 --includes) "
               f"mesh_processor.cpp -o {mesh_processor}", cwd=renderer_dir)
            shutil.copy2(os.path.join(renderer_dir, mesh_processor), cached_processor)
        except RuntimeError:
            print("mesh_processor build failed; using the slower Python fallback.")
    else:
        print("Using cached", cached_processor)
        shutil.copy2(cached_processor, renderer_dir)
    open(MARK, "w").close()

# Check every import in a fresh process, exactly as the stage processes will load them.
check = ("import sys; sys.path[:0] = [%r, %r, %r]; import torch, custom_rasterizer, hy3dgen.shapegen, hy3dgen.texgen, rembg; "
         "from mvadapter.pipelines.pipeline_mvadapter_i2mv_sdxl import MVAdapterI2MVSDXLPipeline; "
         "from scripts.inference_i2mv_sdxl import run_pipeline; import imageto3d.run; "
         "print('torch', torch.__version__, 'pipelines OK')" % (PIPELINE_DIR, MV_REPO, REPO))
result = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
print(result.stdout, result.stderr[-3000:])
if result.returncode != 0:
    os.remove(MARK)
    raise RuntimeError("Pipeline import check failed. Send the last 30 lines above for diagnosis.")

if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)
import importlib
import imageto3d.config, imageto3d.assets, imageto3d.run
for module in (imageto3d.config, imageto3d.assets, imageto3d.run):
    importlib.reload(module)
imageto3d.config.Config(**SETTINGS, hunyuan_repo=REPO, mv_repo=MV_REPO).save(CONFIG_PATH)
print("Install OK. Config written to", CONFIG_PATH)

In [ ]:
# @title 4. Generate 3D for every image in Drive/in
# @markdown Stages run in separate processes (views, shape, paint, export), so RAM and GPU memory are fully freed
# @markdown between stages. A failing image is retried once, then skipped; the rest of the batch continues.
if "CONFIG_PATH" not in globals() or not os.path.exists(CONFIG_PATH):
    raise RuntimeError("Run blocks 1, 2 and 3 first.")
import imageto3d.assets, imageto3d.config, imageto3d.run

config = imageto3d.config.load(CONFIG_PATH)
queue = imageto3d.assets.pending(config)
print(f"{len(queue)} front image(s) waiting:", ", ".join(asset.key for asset in queue) or "none")
imageto3d.run.main(CONFIG_PATH, watch=WATCH)

In [ ]:
# @title 5. Review results and download (optional)
DOWNLOAD_ZIP = False  # @param {type:"boolean"}
import glob, os, shutil, time
from IPython.display import Image as ShowImage, display
import imageto3d.config

config = imageto3d.config.load(CONFIG_PATH)
outputs = sorted(glob.glob(os.path.join(config.out_dir, "*.glb")), key=os.path.getmtime)
for glb in outputs[-10:]:
    print(f"{os.path.basename(glb):55s} {os.path.getsize(glb) / 1024**2:5.1f} MB")
    sheet = glb[:-4] + "-views.jpg"
    if os.path.exists(sheet):
        display(ShowImage(filename=sheet, width=768))
for status_path in glob.glob(os.path.join(config.work_dir, "*", "status.json")):
    status = open(status_path).read()
    if '"last_error": "' in status:
        print("Problem:", os.path.basename(os.path.dirname(status_path)), status)
if DOWNLOAD_ZIP and outputs:
    from google.colab import files
    archive = shutil.make_archive(f"/content/hunyuan3d-{time.strftime('%Y%m%d-%H%M')}", "zip", config.out_dir)
    files.download(archive)

## 出問題時

| 狀況 | 做法 |
|---|---|
| 區塊 1 回報 Python 版本不符 | 「執行階段 → 變更執行階段類型 → 執行階段版本」選 2026.07，再從區塊 1 開始 |
| 2026.07 已從選單消失 | Colab 只保留一段時間的舊版本；需改用新版 runtime 並更新區塊 1 的版本檢查與區塊 3 的 `PINNED` |
| 區塊 3 安裝失敗 | 查看最後 30 行錯誤；安裝會保留 Colab 的 torch/numpy 版本 |
| 區塊 3 編譯的擴充失效 | 刪除 `DRIVE_ROOT/wheels/` 下對應的資料夾後，重開 runtime 重跑區塊 1、2、3 |
| 某張圖顯示 SKIPPED | 那張圖失敗兩次被跳過，其他圖照常處理；看訊息中的錯誤，修正後刪除 `work/<類別>-<id>/status.json` 再跑區塊 4 |
| 顯示 killed: out of memory | 該階段的程序被系統終止，記憶體已隨程序釋放；同一張圖會再試一次。常發生時：區塊 2 的 `VIEW_COUNT` 用 4、`TEXTURE_SIZE` 改 1024，或 `SHAPE_MODEL` 改 turbo |
| 401/403 找不到模型 | 確認已接受 Hugging Face 模型條款；需要時在 Colab Secrets 設 `HF_TOKEN` |
| 某一個視角不理想 | 改區塊 2 的 `VIEW_SEED`，刪除 `work/<類別>-<id>/` 資料夾並把 `in/used-...-front` 改回原檔名，再跑區塊 2～4 |
| 檔名沒被處理 | 第一個字要是 character、monster、armor、appearance、weapon、object、bigobject（或 equipment、prop、environment），結尾是 `-front`、`-stuff` 或 `-building`；多視角的額外圖結尾是 `-left`、`-back`、`-right`，而且要有同 id 的主圖，例如 `weapon-initiate_dagger-stuff.png` |
| 形狀前後顛倒或左右混亂 | 檢查額外圖的方向：left 圖的正面朝向畫面左邊，right 圖朝向右邊；放錯就互換檔名 |
| 顯示 WAITING … missing | bigobject 四張圖沒放齊（主圖要叫 `-front`，不是 `-building`）；補上後會自動開始 |
| bigobject 很慢 | 正常：形狀網格 512、20 萬面、4K 貼圖，每棟可能要 15～30 分鐘；顯示 retrying 表示 T4 放不下，已自動降一級 |
| 髮型模型連著假人頭 | 單張圖會把圖裡看得到的東西都建出來；髮型圖請用透明背景、不含假人頭，或之後在 Blender 刪掉頭部 |
| 頭髮在背面消失或不一致 | 用光頭的正面圖（見最上方的 Gemini 提示詞），`HEAD_MODE` 維持 `bald` |
| Colab 斷線 | 重開後跑區塊 1～4；做到一半的圖會從中斷的階段接著做（中間檔在 `DRIVE_ROOT/work`） |